# Introduction

In this notebook, we will do a comprehensive analysis of the Android app market by comparing thousands of apps in the Google Play store.

# About the Dataset of Google Play Store Apps & Reviews

**Data Source:** <br>
App and review data was scraped from the Google Play Store by Lavanya Gupta in 2018. Original files listed [here](
https://www.kaggle.com/lava18/google-play-store-apps).

# Import Statements

In [1]:
import pandas as pd
import plotly.express as px

# Notebook Presentation

In [2]:
# Show numeric output in decimal format e.g., 2.15
pd.options.display.float_format = '{:,.2f}'.format

# Read the Dataset

In [3]:
df_apps = pd.read_csv('apps.csv')

# Data Cleaning

**Challenge**: How many rows and columns does `df_apps` have? What are the column names? Look at a random sample of 5 different rows with [.sample()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sample.html).

In [4]:
df_apps.shape

(10841, 12)

In [5]:
for column in df_apps.columns:
    print(df_apps[column].name)

App
Category
Rating
Reviews
Size_MBs
Installs
Type
Price
Content_Rating
Genres
Last_Updated
Android_Ver


In [6]:
df_apps.sample(5)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Last_Updated,Android_Ver
603,BU Beauty Rooms,BEAUTY,NaN,1,6.00,50,Free,0,Everyone,Beauty,"February 17, 2017",4.4 and up
1966,Photo BG Changer,PHOTOGRAPHY,3.80,11,17.00,"1,000",Free,0,Everyone,Photography,"February 21, 2018",4.1 and up
5405,Real Estate by Movoto,LIFESTYLE,4.40,3114,13.00,"100,000",Free,0,Everyone,Lifestyle,"February 20, 2018",4.0.3 and up
7201,SIM Card Info,TOOLS,4.10,4234,1.90,"1,000,000",Free,0,Everyone,Tools,"May 25, 2018",4.0 and up
5482,Webmail web mobile app,TOOLS,2.30,395,2.60,"100,000",Free,0,Everyone,Tools,"August 5, 2018",4.1 and up


### Drop Unused Columns

**Challenge**: Remove the columns called `Last_Updated` and `Android_Version` from the DataFrame. We will not use these columns. 

In [7]:
df_apps = df_apps.drop(columns=["Last_Updated", "Android_Ver"])
print(df_apps.head())

                                            App         Category  Rating  \
0                       Ak Parti Yardım Toplama           SOCIAL     NaN   
1                    Ain Arabic Kids Alif Ba ta           FAMILY     NaN   
2  Popsicle Launcher for Android P 9.0 launcher  PERSONALIZATION     NaN   
3                     Command & Conquer: Rivals           FAMILY     NaN   
4                                    CX Network         BUSINESS     NaN   

   Reviews  Size_MBs Installs  Type   Price Content_Rating           Genres  
0        0      8.70        0  Paid  $13.99           Teen           Social  
1        0     33.00        0  Paid   $2.99       Everyone        Education  
2        0      5.50        0  Paid   $1.49       Everyone  Personalization  
3        0     19.00        0   NaN       0   Everyone 10+         Strategy  
4        0     10.00        0  Free       0       Everyone         Business  


### Find and Remove NaN values in Ratings

**Challenge**: How may rows have a NaN value (not-a-number) in the Ratings column? Create DataFrame called `df_apps_clean` that does not include these rows. 

In [8]:
count_nan_ratings = df_apps['Rating'].isna().sum()
print("NaN values in Ratings column:",count_nan_ratings)

NaN values in Ratings column: 1474


In [9]:
df_apps_clean = df_apps.dropna(subset=['Rating'])
print("Shape of df_apps_clean:", df_apps_clean.shape)

Shape of df_apps_clean: (9367, 10)


### Find and Remove Duplicates

**Challenge**: Are there any duplicates in data? Check for duplicates using the [.duplicated()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.duplicated.html) function. How many entries can you find for the "Instagram" app? Use [.drop_duplicates()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html) to remove any duplicates from `df_apps_clean`. 


In [10]:
print("Duplicates in df_apps_clean:", df_apps_clean.duplicated().sum())

Duplicates in df_apps_clean: 476


In [11]:
df_apps_clean = df_apps_clean.drop_duplicates()

In [12]:
print("Shape of df_apps_clean after dropping duplicates:", df_apps_clean.shape)

Shape of df_apps_clean after dropping duplicates: (8891, 10)


# Find Highest Rated Apps

**Challenge**: Identify which apps are the highest rated. What problem might you encounter if you rely exclusively on ratings alone to determine the quality of an app?

In [13]:
highest_rated_apps = df_apps_clean.sort_values(by='Rating', ascending=False)
print("Highest Rated Apps:\n", highest_rated_apps.head(5))

Highest Rated Apps:
                         App Category  Rating  Reviews  Size_MBs Installs  \
186  Basics of Orthopaedics  MEDICAL    5.00        1      5.60       10   
21      KBA-EZ Health Guide  MEDICAL    5.00        4     25.00        1   
28                 Ra Ga Ba     GAME    5.00        2     20.00        1   
47                  Mu.F.O.     GAME    5.00        2     16.00        1   
237                EC Mover     GAME    5.00        5      4.60       10   

     Type  Price Content_Rating   Genres  
186  Free      0       Everyone  Medical  
21   Free      0       Everyone  Medical  
28   Paid  $1.49       Everyone   Arcade  
47   Paid  $0.99       Everyone   Arcade  
237  Free      0       Everyone   Racing  


# Find 5 Largest Apps in terms of Size (MBs)

**Challenge**: What's the size in megabytes (MB) of the largest Android apps in the Google Play Store. Based on the data, do you think there could be limit in place or can developers make apps as large as they please? 

In [14]:
largest_apps = df_apps_clean.nlargest(5, 'Size_MBs')
print("5 Largest Apps by Size:\n", largest_apps[['App', 'Size_MBs']])

5 Largest Apps by Size:
                                                    App  Size_MBs
1795                              Navi Radiography Pro    100.00
3144                                        Vi Trainer    100.00
4176  Car Crash III Beam DH Real Damage Simulator 2018    100.00
7926                                         Post Bank    100.00
7927                       The Walking Dead: Our World    100.00


# Find the 5 App with Most Reviews

**Challenge**: Which apps have the highest number of reviews? Are there any paid apps among the top 50?

In [15]:
most_reviewed_apps = df_apps_clean.sort_values(by='Reviews', ascending=False).head(50)
print("5 Apps with Most Reviews:\n", most_reviewed_apps.head(5))
found = False
for type in most_reviewed_apps['Type']:
    if type == 'Paid':
        print("There are paid apps in the most reviewed apps list.")
        break
if not found:
    print("There are no paid apps in the most reviewed apps list.")

5 Apps with Most Reviews:
                       App       Category  Rating   Reviews  Size_MBs  \
10805            Facebook         SOCIAL    4.10  78158306      5.30   
10811            Facebook         SOCIAL    4.10  78128208      5.30   
10785  WhatsApp Messenger  COMMUNICATION    4.40  69119316      3.50   
10797  WhatsApp Messenger  COMMUNICATION    4.40  69109672      3.50   
10808           Instagram         SOCIAL    4.50  66577446      5.30   

            Installs  Type Price Content_Rating         Genres  
10805  1,000,000,000  Free     0           Teen         Social  
10811  1,000,000,000  Free     0           Teen         Social  
10785  1,000,000,000  Free     0       Everyone  Communication  
10797  1,000,000,000  Free     0       Everyone  Communication  
10808  1,000,000,000  Free     0           Teen         Social  
There are no paid apps in the most reviewed apps list.


# Plotly Pie and Donut Charts - Visualise Categorical Data: Content Ratings

In [16]:
ratings = df_apps_clean.Content_Rating.value_counts()
ratings

Content_Rating
Everyone           7094
Teen               1022
Mature 17+          411
Everyone 10+        360
Adults only 18+       3
Unrated               1
Name: count, dtype: int64

In [17]:
fig = px.pie(labels=ratings.index,values=ratings.values,title="Content Rating",names=ratings.index,)

In [18]:
fig.update_traces(textposition='outside', textinfo='percent+label')

In [19]:
fig.show()

# Numeric Type Conversion: Examine the Number of Installs

**Challenge**: How many apps had over 1 billion (that's right - BILLION) installations? How many apps just had a single install? 

Check the datatype of the Installs column.

Count the number of apps at each level of installations. 

Convert the number of installations (the Installs column) to a numeric data type. Hint: this is a 2-step process. You'll have make sure you remove non-numeric characters first. 

In [20]:
df_apps_clean.Installs.describe()

count          8891
unique           19
top       1,000,000
freq           1486
Name: Installs, dtype: object

In [21]:
df_apps_clean.info()

<class 'pandas.DataFrame'>
Index: 8891 entries, 21 to 10840
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             8891 non-null   str    
 1   Category        8891 non-null   str    
 2   Rating          8891 non-null   float64
 3   Reviews         8891 non-null   int64  
 4   Size_MBs        8891 non-null   float64
 5   Installs        8891 non-null   str    
 6   Type            8891 non-null   str    
 7   Price           8891 non-null   str    
 8   Content_Rating  8891 non-null   str    
 9   Genres          8891 non-null   str    
dtypes: float64(2), int64(1), str(7)
memory usage: 764.1 KB


In [22]:
# Installs of data type object because of the comma (,) character
df_apps_clean[['App', 'Installs']].groupby('Installs').count()

,App
Installs,
1,3
"1,000",699
"1,000,000",1486
"1,000,000,000",49
10,69
"10,000",989
"10,000,000",1130
100,303
"100,000",1110


In [23]:
df_apps_clean.Installs = df_apps_clean.Installs.astype(str).str.replace(',', "")

df_apps_clean.Installs = pd.to_numeric(df_apps_clean.Installs)

df_apps_clean[['App', 'Installs']].groupby('Installs').count()

,App
Installs,
1,3
5,9
10,69
50,56
100,303
500,199
1000,699
5000,426
10000,989


# Find the Most Expensive Apps, Filter out the Junk, and Calculate a (ballpark) Sales Revenue Estimate

Let's examine the Price column more closely.

**Challenge**: Convert the price column to numeric data. Then investigate the top 20 most expensive apps in the dataset.

Remove all apps that cost more than $250 from the `df_apps_clean` DataFrame.

Add a column called 'Revenue_Estimate' to the DataFrame. This column should hold the price of the app times the number of installs. What are the top 10 highest grossing paid apps according to this estimate? Out of the top 10 highest grossing paid apps, how many are games?


In [24]:
df_apps_clean.Price = df_apps_clean.Price.astype(str).str.replace('$', "")
df_apps_clean.Price = pd.to_numeric(df_apps_clean.Price)

df_apps_clean.sort_values('Price', ascending=False).head(20)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
3946,I'm Rich - Trump Edition,LIFESTYLE,3.60,275,7.30,10000,Paid,400.00,Everyone,Lifestyle
2775,I Am Rich Pro,FAMILY,4.40,201,2.70,5000,Paid,399.99,Everyone,Entertainment
4606,I Am Rich Premium,FINANCE,4.10,1867,4.70,50000,Paid,399.99,Everyone,Finance
3114,I am Rich,FINANCE,4.30,180,3.80,5000,Paid,399.99,Everyone,Finance
5765,I am rich,LIFESTYLE,3.80,3547,1.80,100000,Paid,399.99,Everyone,Lifestyle
2394,I am Rich!,FINANCE,3.80,93,22.00,1000,Paid,399.99,Everyone,Finance
2461,I AM RICH PRO PLUS,FINANCE,4.00,36,41.00,1000,Paid,399.99,Everyone,Finance
3145,I am rich(premium),FINANCE,3.50,472,0.94,5000,Paid,399.99,Everyone,Finance
1331,most expensive app (H),FAMILY,4.30,6,1.50,100,Paid,399.99,Everyone,Entertainment
1946,I am rich (Most expensive app),FINANCE,4.10,129,2.70,1000,Paid,399.99,Teen,Finance


### The most expensive apps sub $250

In [25]:
df_apps_clean = df_apps_clean[df_apps_clean['Price'] < 250]
df_apps_clean.sort_values('Price', ascending=False).head(5)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
2281,Vargo Anesthesia Mega App,MEDICAL,4.60,92,32.00,1000,Paid,79.99,Everyone,Medical
1407,LTC AS Legal,MEDICAL,4.00,6,1.30,100,Paid,39.99,Everyone,Medical
2629,I am Rich Person,LIFESTYLE,4.20,134,1.80,1000,Paid,37.99,Everyone,Lifestyle
2481,A Manual of Acupuncture,MEDICAL,3.50,214,68.00,1000,Paid,33.99,Everyone,Medical
2463,PTA Content Master,MEDICAL,4.20,64,41.00,1000,Paid,29.99,Everyone,Medical


### Highest Grossing Paid Apps (ballpark estimate)

In [26]:
df_apps_clean['Revenue_Estimate'] = df_apps_clean.Installs.mul(df_apps_clean.Price)
df_apps_clean.sort_values('Revenue_Estimate', ascending=False)[:10]

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue_Estimate
9220,Minecraft,FAMILY,4.50,2376564,19.00,10000000,Paid,6.99,Everyone 10+,Arcade;Action & Adventure,"69,900,000.00"
9224,Minecraft,FAMILY,4.50,2375336,19.00,10000000,Paid,6.99,Everyone 10+,Arcade;Action & Adventure,"69,900,000.00"
8825,Hitman Sniper,GAME,4.60,408292,29.00,10000000,Paid,0.99,Mature 17+,Action,"9,900,000.00"
7151,Grand Theft Auto: San Andreas,GAME,4.40,348962,26.00,1000000,Paid,6.99,Mature 17+,Action,"6,990,000.00"
7977,Sleep as Android Unlock,LIFESTYLE,4.50,23966,0.85,1000000,Paid,5.99,Everyone,Lifestyle,"5,990,000.00"
7477,Facetune - For Free,PHOTOGRAPHY,4.40,49553,48.00,1000000,Paid,5.99,Everyone,Photography,"5,990,000.00"
6594,DraStic DS Emulator,GAME,4.60,87766,12.00,1000000,Paid,4.99,Everyone,Action,"4,990,000.00"
6082,Weather Live,WEATHER,4.50,76593,4.75,500000,Paid,5.99,Everyone,Weather,"2,995,000.00"
7954,Bloons TD 5,FAMILY,4.60,190086,94.00,1000000,Paid,2.99,Everyone,Strategy,"2,990,000.00"
7633,Five Nights at Freddy's,GAME,4.60,100805,50.00,1000000,Paid,2.99,Teen,Action,"2,990,000.00"


# Plotly Bar Charts & Scatter Plots: Analysing App Categories

In [27]:
# Number of different categories
df_apps_clean.Category.nunique()

33

In [28]:
# Number of apps per category
top10_category = df_apps_clean.Category.value_counts()[:10]
top10_category

Category
FAMILY             1714
GAME               1074
TOOLS               733
PRODUCTIVITY        334
FINANCE             311
PERSONALIZATION     310
COMMUNICATION       307
PHOTOGRAPHY         304
MEDICAL             302
LIFESTYLE           301
Name: count, dtype: int64

### Vertical Bar Chart - Highest Competition (Number of Apps)

In [29]:
bar = px.bar(
        x = top10_category.index, # index = category name
        y = top10_category.values)

bar.show()

### Horizontal Bar Chart - Most Popular Categories (Highest Downloads)

In [31]:
category_installs = df_apps_clean.groupby('Category').agg({'Installs': pd.Series.sum})
category_installs.sort_values('Installs', ascending=True, inplace=True)

In [32]:
h_bar = px.bar(x=category_installs.Installs,
               y=category_installs.index,
               orientation='h',
               title='Category Popularity')

h_bar.update_layout(xaxis_title='Number of Downloads', yaxis_title='Category')
h_bar.show()

### Category Concentration - Downloads vs. Competition

**Challenge**: 
* First, create a DataFrame that has the number of apps in one column and the number of installs in another:

<img src=https://imgur.com/uQRSlXi.png width="350">

* Then use the [plotly express examples from the documentation](https://plotly.com/python/line-and-scatter/) alongside the [.scatter() API reference](https://plotly.com/python-api-reference/generated/plotly.express.scatter.html)to create scatter plot that looks like this. 

<img src=https://imgur.com/cHsqh6a.png>

*Hint*: Use the size, hover_name and color parameters in .scatter(). To scale the yaxis, call .update_layout() and specify that the yaxis should be on a log-scale like so: yaxis=dict(type='log') 

In [34]:
df_by_category = df_apps_clean.groupby('Category').agg({'App': pd.Series.count})
df_by_category = pd.merge(df_by_category, category_installs, on='Category', how="inner")
print(f'The dimensions of the DataFrame are: {df_by_category.shape}')
df_by_category.sort_values('Installs', ascending=False)

The dimensions of the DataFrame are: (33, 2)


,App,Installs
Category,,
GAME,1074,31543862717
COMMUNICATION,307,24152241530
SOCIAL,244,12513841475
PRODUCTIVITY,334,12463070180
TOOLS,733,11440724500
FAMILY,1714,10041105490
PHOTOGRAPHY,304,9721243130
TRAVEL_AND_LOCAL,205,6361859300
VIDEO_PLAYERS,160,6221897200


In [35]:
scatter = px.scatter(df_by_category, # data
                    x='App', # column name
                    y='Installs',
                    title='Category Concentration',
                    size='App',
                    hover_name=df_by_category.index,
                    color='Installs')

scatter.update_layout(xaxis_title="Number of Apps (Lower=More Concentrated)",
                      yaxis_title="Installs",
                      yaxis=dict(type='log'))

scatter.show()

# Extracting Nested Data from a Column

**Challenge**: How many different types of genres are there? Can an app belong to more than one genre? Check what happens when you use .value_counts() on a column with nested values? See if you can work around this problem by using the .split() function and the DataFrame's [.stack() method](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html). 


In [42]:
# Number of Genres?
len(df_apps_clean.Genres.unique())

115

In [41]:
# Problem: Have multiple categories seperated by ;
df_apps_clean.Genres.value_counts().sort_values(ascending=True)[:5]

Genres
Tools;Education               1
Role Playing;Brain Games      1
Lifestyle;Pretend Play        1
Health & Fitness;Education    1
Adventure;Brain Games         1
Name: count, dtype: int64

In [43]:
# Split the strings on the semi-colon and then .stack them.
stack = df_apps_clean.Genres.str.split(';', expand=True).stack()
print(f'We now have a single column with shape: {stack.shape}')
num_genres = stack.value_counts()
print(f'Number of genres: {len(num_genres)}')

We now have a single column with shape: (17752,)
Number of genres: 53


# Colour Scales in Plotly Charts - Competition in Genres

**Challenge**: Can you create this chart with the Series containing the genre data? 

<img src=https://imgur.com/DbcoQli.png width=400>

Try experimenting with the built in colour scales in Plotly. You can find a full list [here](https://plotly.com/python/builtin-colorscales/). 

* Find a way to set the colour scale using the color_continuous_scale parameter. 
* Find a way to make the color axis disappear by using coloraxis_showscale. 

In [46]:
genres_bar= px.bar(num_genres,
                            x=num_genres.index,
                            y=num_genres.values,
                            title='Top Genres',
                            hover_name=num_genres.index,
                            color=num_genres.values,
                            color_continuous_scale='Agsunset')

genres_bar.update_layout(xaxis_title='Genres',yaxis_title='Number of Apps')

genres_bar.show()

# Grouped Bar Charts: Free vs. Paid Apps per Category

In [47]:
df_apps_clean.Type.value_counts()

Type
Free    8278
Paid     598
Name: count, dtype: int64

In [48]:
df_free_vs_paid = df_apps_clean.groupby(["Category", "Type"],
                                      as_index=False).agg({'App': pd.Series.count})
df_free_vs_paid.sort_values('App')

,Category,Type,App
3,AUTO_AND_VEHICLES,Paid,1
17,ENTERTAINMENT,Paid,2
24,FOOD_AND_DRINK,Paid,2
38,NEWS_AND_MAGAZINES,Paid,2
48,SHOPPING,Paid,2
...,...,...,...
21,FINANCE,Free,304
45,PRODUCTIVITY,Free,316
53,TOOLS,Free,670
25,GAME,Free,997


**Challenge**: Use the plotly express bar [chart examples](https://plotly.com/python/bar-charts/#bar-chart-with-sorted-or-ordered-categories) and the [.bar() API reference](https://plotly.com/python-api-reference/generated/plotly.express.bar.html#plotly.express.bar) to create this bar chart: 

<img src=https://imgur.com/LE0XCxA.png>

You'll want to use the `df_free_vs_paid` DataFrame that you created above that has the total number of free and paid apps per category. 

See if you can figure out how to get the look above by changing the `categoryorder` to 'total descending' as outlined in the documentation here [here](https://plotly.com/python/categorical-axes/#automatically-sorting-categories-by-name-or-total-value). 

In [49]:
paid_free_bar = px.bar(df_free_vs_paid,
                       x='Category',
                       y='App',
                       title='Free vs Paid Apps by Category',
                       color='Type',
                       barmode='group')

paid_free_bar.update_layout(xaxis_title='Category',
                    yaxis_title='Number of Apps',
                    xaxis={'categoryorder':'total descending'},
                    yaxis=dict(type='log'))

paid_free_bar.show()

# Plotly Box Plots: Lost Downloads for Paid Apps

**Challenge**: Create a box plot that shows the number of Installs for free versus paid apps. How does the median number of installations compare? Is the difference large or small?

Use the [Box Plots Guide](https://plotly.com/python/box-plots/) and the [.box API reference](https://plotly.com/python-api-reference/generated/plotly.express.box.html) to create the following chart. 

<img src=https://imgur.com/uVsECT3.png>


In [65]:
box = px.box(df_apps_clean,
             y='Installs',
             x='Type',
             color='Type',
             notched=True,
             points='all',
             title='How Many Downloads are Paid Apps Giving Up?')

box.update_layout(yaxis=dict(type='log'))

box.show()

# Plotly Box Plots: Revenue by App Category

**Challenge**: See if you can generate the chart below: 

<img src=https://imgur.com/v4CiNqX.png>

Looking at the hover text, how much does the median app earn in the Tools category? If developing an Android app costs $30,000 or thereabouts, does the average photography app recoup its development costs?

Hint: I've used 'min ascending' to sort the categories.

In [66]:
df_paid_apps = df_apps_clean[df_apps_clean['Type'] == 'Paid']

In [67]:
box = px.box(df_paid_apps,
             x='Category',
             y='Revenue_Estimate',
             title='How Much Can Paid Apps Earn?')

box.update_layout(xaxis_title='Category',
                  yaxis_title='Paid App Ballpark Revenue',
                  xaxis={'categoryorder': 'min ascending'},
                  yaxis=dict(type='log'))

box.show()

# How Much Can You Charge? Examine Paid App Pricing Strategies by Category

**Challenge**: What is the median price price for a paid app? Then compare pricing by category by creating another box plot. But this time examine the prices (instead of the revenue estimates) of the paid apps. I recommend using `{categoryorder':'max descending'}` to sort the categories.

In [54]:
df_paid_apps = df_apps_clean[df_apps_clean['Type'] == 'Paid']

In [68]:
price_per_category_box = px.box(df_paid_apps,
             x='Category',
             y="Price",
             title='Price per Category')

price_per_category_box.update_layout(xaxis_title='Category',
                  yaxis_title='Paid App Price',
                  xaxis={'categoryorder':'max descending'},
                  yaxis=dict(type='log'))


price_per_category_box.show()